__Prerequisites:__

* Access to the [`bdcmsg22/QC_metabolomics`](https://app.terra.bio/#workspaces/bdcmsg22/QC_metabolomics) workspace where the raw metabolomics data is stored.

In [ ]:
library(data.table)
library(matrixStats)
'%ni%' <- Negate('%in%')

options(datatable.na.strings=c('NA',''))

dir.create("data/raw/metabolomics",             rec=T)
dir.create("data/derived/metabolomics/cleaned", rec=T)
dir.create("data/derived/metabolomics/QCd",     rec=T)

data_csvs <- list(CP='data/raw/metabolomics/23_0207_MESA_Pilot_X01_Broad_C8-pos.csv',    # C8-positive
                  CN='data/raw/metabolomics/23_0210_MESA_Pilot_X01_Broad_C18-neg.csv',   # C18-negative
                  HP='data/raw/metabolomics/23_0207_MESA_Pilot_X01_Broad_HILIC-pos.csv', # HILIC-positive
                  AN='data/raw/metabolomics/23_0210_MESA_Pilot_X01_BIDMC_Amide-neg.csv') # Amide-negative

# Download original xlsx data and convert to csv, faster to work with.
# It's extremely slow to read from xlsx, but this only needs to run once and then we can use the faster csv.
if(!all(file.exists(unlist(data_csvs)))) {
  system("gcloud storage cp gs://fc-secure-4b3e979d-ba8b-43c4-a5ec-b41ab42ce606/MesaMetabolomics_PilotX01_2023_02_16/23_02* data/raw/metabolomics") # Workspace: bdcmsg22/QC_metabolomics
  data_xlsxs <- sub('.csv','.xlsx', data_csvs)
  Map(data_csvs,data_xlsxs, f=\(csv,xlsx)
    xlsx |> readxl::read_xlsx(col_names=F) |> fwrite(csv, col.names=F) )
}

# Inspect raw data
There are files for 4 LC-MS methods: C8-positive, C18-negative, HILIC-positive, Amide-negative ([descriptions here](https://topmed.nhlbi.nih.gov/sites/default/files/TOPMed_CORE_Year3_Broad-BIDMC_metabolomics_methods.pdf)).

In [ ]:
quick_look <- \(file) fread(file, nrow=12, select=1:12) |> knitr::kable()
lapply(data_csvs, quick_look)

Nonstandard format, so prepare for hardcoded row/col numbers!

# Extract sample metadata
Note: MESA AN file has no sample metadata, but all its sample IDs are covered by CP and HP methods anyways.

In [ ]:
sample_info <- list()
sample_info$CP <- fread(data_csvs$CP, nrow=10, drop=1:8) |> transpose()
sample_info$CN <- fread(data_csvs$CN, nrow=10, drop=1:8) |> transpose()
sample_info$HP <- fread(data_csvs$HP, nrow=10, drop=1:8) |> transpose()
sample_info$AN <- fread(data_csvs$AN, nrow= 9, drop=1:8) |> transpose()

names(sample_info$CP) <- names(sample_info$CN) <- names(sample_info$HP) <- c('extr_date','inject_date','batch','inject_order','sample_type',           'TOM_Id','checksum','raw_fnm','project','sample_id')
names(sample_info$AN) <-                                                   c(            'inject_date','batch','inject_order','sample_type','BIDMC_Id','TOM_Id',           'raw_fnm','project','sample_id')

by_cols <- c('sample_id','TOM_Id','project')
sample_info <- sample_info$CP |>
       merge(y=sample_info$CN, suffixes=c('','_cn'), by=by_cols, all=T) |>
       merge(y=sample_info$HP, suffixes=c('','_hp'), by=by_cols, all=T) |>
       merge(y=sample_info$AN, suffixes=c('','_an'), by=by_cols, all=T) |>
       setnames(c('extr_date',   'inject_date',   'batch',    'inject_order',   'sample_type',   'checksum',   'raw_fnm'   ),
                c('extr_date_cp','inject_date_cp','batch_cp','inject_order_cp','sample_type_cp','checksum_cp','raw_fnm_cp'))

#sample_info[duplicated(sample_id),sample_id] # None of the duplicated IDs are TOM ids.

sample_info[, is_qc_sample := !grepl("TOM",sample_id)]

# Dates were converted to integers in readxl::read_xlsx. This integer represents the # of days after 1899-12-30 (see ?as.Date.)
date_cols <- grep('date',names(sample_info),value=T)
sample_info[, (date_cols) := lapply(.SD, \(x) as.numeric(x) |> as.Date(origin='1899-12-30')), .SDcols=date_cols]

# Separate amine batch into batch & plate number
sample_info[, plate_an := sub('B[0-9]+','', batch_an)]
sample_info[, batch_an := sub('P[0-9]+','', batch_an)]

fwrite(sample_info, "data/derived/metabolomics/sample_info-mesa.csv")
head(sample_info)

# Extract metabolite metadata
Then, create a file mapping these IDs back to their metabolite metadata such as M/Z and RT.

In [ ]:
met_info <- list()
met_info$CP <- fread(data_csvs$CP, skip=9, select=1:8)
met_info$CN <- fread(data_csvs$CN, skip=9, select=1:8)
met_info$HP <- fread(data_csvs$HP, skip=9, select=1:8)
met_info$AN <- fread(data_csvs$AN, skip=8, select=1:8)

# Temporarily renames AN's MRM column to RT, for consistent colnames for rbindlist.
names(met_info$AN) <- names(met_info$CP) 
met_info <- rbindlist(met_info)
met_info[Method=='Amide-neg', `:=`(MRM_Transition=RT, RT=NA)]

# Compound IDs may be duplicated across CP/CN/HP/AN. A concatenated `<ID>_<Method>` variable ensures IDs are unique.
met_info[,                    unique_met_id := paste0(Compound_ID,'_',sub('-','_',Method))] # R doesn't like '-' in names.
met_info[Method=='Amide-neg', unique_met_id := paste0('met', .I,  '_',sub('-','_',Method))] # Amines have no Compound_ID, so generate them.
#sum(duplicated(met_info$unique_id)) # No dup IDs!

met_info[met_info==''] <- NA

fwrite(met_info, "data/derived/metabolomics/met_info-mesa.csv")
head(met_info)

# Extract measurement data

In [ ]:
data <- list()
data$CP <- fread(data_csvs$CP, skip=9, drop=1:8, colClasses='double') |> t() |> `colnames<-`(met_info[Method==   'C8-pos',unique_met_id]) # I double-checked, the names are in the proper order
data$CN <- fread(data_csvs$CN, skip=9, drop=1:8, colClasses='double') |> t() |> `colnames<-`(met_info[Method==  'C18-neg',unique_met_id])
data$HP <- fread(data_csvs$HP, skip=9, drop=1:8, colClasses='double') |> t() |> `colnames<-`(met_info[Method=='HILIC-pos',unique_met_id])
data$AN <- fread(data_csvs$AN, skip=8, drop=1:8, colClasses='double') |> t() |> `colnames<-`(met_info[Method=='Amide-neg',unique_met_id])

mode(data$CP) <- mode(data$CN) <- mode(data$HP) <- mode(data$AN) <- "numeric" # "NAs introduced by coersion" are due the AN file, which contains to empty strings "" and one weird cell "1+2559:2584.03935869994933".

data <- lapply(data, \(m) m[rownames(m) %ni% sample_info[is_qc_sample==T,sample_id], ] ) # Rm QC sample rows
data$AN[data$AN<0] <- NA # Amines has 11 negative measurements, which don't make sense

Map(fwrite, data, paste0("data/derived/metabolomics/cleaned/",names(data),"_cleaned-mesa.csv"), row.names=T)

In [ ]:
data$CP[1:3,1:3] # Check out the cleaned data!
data$HP[1:3,1:3] # Just simple matrices of sample x metabolite IDs.
data$AN[1:3,1:3]

# QC
1\. Remove signatures w/ variance = 0\
2\. Remove signatures w/ >25% missingness\
3\. Half-min Impute\
4\. Log2\
5\. Winsorize to 5*sd\
Missingness removal is done in two separate steps, where high-missingness metabolites are removed first, _then_ samples (to prioritize not losing samples).\
Batch adjustment is best done at the same time as adjusting for other covariates, later.

In [ ]:
winsorizeColsBySd <- \(m, n_sd) {
  cmeans <- colMeans(m,na.rm=T)
  csds   <- colSds  (m,na.rm=T)
  lower <- cmeans - n_sd*csds
  upper <- cmeans + n_sd*csds
  t(pmin(pmax(t(m),lower),upper)) # lower/upper are vectors of length ncol(m), recycled for each row of the matrix.
}

data <- lapply(data, \(m) {
  m <- m[, colVars(m,na.rm=T) > 0] # step 1
  m <- m[,colSums(is.na(m)) < 0.25*nrow(m) ] # step 2a
  m <- m[ rowSums(is.na(m)) < 0.25*ncol(m),] # step 2b
  m[is.na(m)] <- (colMins(m,na.rm=T)/2)[col(m)[is.na(m)]] # step 3
  m <- log2(m+1) # step 4
  m <- winsorizeColsBySd(m,5) # step 5
})

Map(fwrite, data, paste0('data/derived/metabolomics/QCd/',names(data),'_QCd-mesa.csv'))

# Merge CP/CN/HP/AN datasets
8174/12183 non-QC samples have metabolomics data for all of CP/CN/HP/AN. We still want to preserve those ~4k samples that don't, so we merge the metabolomics data with `all=T`.

In [ ]:
data_dts <- lapply(data, as.data.table, keep.rownames='TOM_Id')
data_merged <- Reduce(\(x,y) merge(x,y, all=T, by='TOM_Id'), data_dts)
fwrite(data_merged, 'data/derived/metabolomics/QCd/merged_QCd-mesa.csv')

---

# Plots
Plots are made using the CP/HP/AN data separately (not merged).

## Distribution of metabolite medians

In [ ]:
par(mfrow=c(1,4))
Map(data, names(data), f = \(m,nm) hist(colMedians(m), main=nm))

## Skew vs. Kurtosis

In [ ]:
plotSkewKurt <- function(df, title) {
  n <- nrow(df); ms <- colMeans(df); sds <- colSds(df)
  skews <- sapply(1:ncol(df),\(col) (sum((df[,col]-ms[col])^3)/sds[col]^3)/n   )
  kurts <- sapply(1:ncol(df),\(col) (sum((df[,col]-ms[col])^4)/sds[col]^4)/n-3 )
  sk <- data.frame(skews=skews, kurts=kurts)

  {plot(sk$skews, sk$kurts, pch = 16, cex = 0.5, main = title, 
    xlab = "Skews", ylab = "Kurts", col = "black") +
   abline(v = -0.5, lty = 2,  lwd = 2) + 
   abline(v =  0.5, lty = 2,  lwd = 2) +
   abline(h = -2.0, lty = 2,  lwd = 2) +
   abline(h =  2.0, lty = 2,  lwd = 2)}
}

par(mfrow=c(1,4))
Map(plotSkewKurt, data, names(data))

## Individual metabolite distributions

In [ ]:
plotAllMets <- function(mtx) {
  par(mar=c(.1,.1,.4,.1), oma=rep(0,4), xaxt='n', yaxt='n', lwd=0.1,
      mfrow = c(ceiling(ncol(mtx)/30), 30)
  )
  sapply(1:ncol(mtx), FUN = \(i)
    plot(mtx[,i], main=colnames(mtx)[i], cex.main=0.4,
         pch=16, col=rgb(0,0,0,0.5), cex=0.1)
  )
}

png("met_dists_CP.png", width = 10000, height =  9000, res=800)
plotAllMets(data$CP)
dev.off()
png("met_dists_CN.png", width = 10000, height = 14000, res=800)
plotAllMets(data$CN)
dev.off()
png("met_dists_HP.png", width = 10000, height =  8000, res=800)
plotAllMets(data$HP)
dev.off()
png("met_dists_AN.png", width = 10000, height =  1200, res=800)
plotAllMets(data$AN)
dev.off()

![](met_dists_CP.png)

![](met_dists_CN.png)

![](met_dists_HP.png)

![](met_dists_AN.png)